In [ ]:
! pip install groq pandas

import os
from groq import Groq
import pandas as pd

# Configure ta clé API Groq ici pour les tests du notebook
GROQ_API_KEY = "gsk_ta_cle_secrete_ici"
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def tester_generation_rapport(datos_cliente, resultado_ml, probabilidad, temperature=0.4):
    """
    Fonction de test pour évaluer les réponses de Llama 3 via Groq
    avec différentes températures et profils clients.
    """
    prompt_usuario = f"""
    Actúa como un analista de riesgo financiero experto.
    Un cliente solicita un préstamo con el siguiente perfil:
    - Edad: {datos_cliente['Edad']} años
    - Ingresos Anuales: ${datos_cliente['Ingresos']}
    - Monto Solicitado: ${datos_cliente['Monto']}
    - Relación Deuda-Ingreso (DTI): {datos_cliente['DTI']}
    - Cuentas de Ahorro: {datos_cliente['Ahorros']}
    - Cuenta Corriente: {datos_cliente['Corriente']}
    
    Nuestro modelo predictivo ha determinado que el cliente es de **{resultado_ml}** con una probabilidad de incumplimiento del {probabilidad*100:.1f}%.
    
    Redacta un reporte profesional de 2 a 3 párrafos explicando la decisión basada en estos factores y proporciona una recomendación clara a la entidad financiera.
    """
    
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": "Eres un asistente experto en análisis de riesgo crediticio para entidades bancarias."},
                {"role": "user", "content": prompt_usuario}
            ],
            model="llama-3.1-8b-instant",
            temperature=temperature,
            max_tokens=500,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"❌ Erreur : {e}"

In [ ]:
# Profil 1 : Excellent profil (Risque Faible)
client_excellent = {
    "Edad": 45, "Ingresos": 120000, "Monto": 20000, 
    "DTI": 0.15, "Ahorros": "rich", "Corriente": "rich"
}

# Profil 2 : Profil très risqué (Jeune, faible revenu, grosse demande)
client_critique = {
    "Edad": 21, "Ingresos": 18000, "Monto": 75000, 
    "DTI": 0.75, "Ahorros": "little", "Corriente": "little"
}

In [ ]:
print("==================================================")
print("TEST 1 : CLIENT EXCELLENT (Température basse = 0.2)")
print("==================================================")
rapport_top = tester_generation_rapport(client_excellent, "Bajo Riesgo (Good)", 0.04, temperature=0.2)
print(rapport_top)

print("\n==================================================")
print("TEST 2 : CLIENT CRITIQUE (Température basse = 0.2)")
print("==================================================")
rapport_critique = tester_generation_rapport(client_critique, "Alto Riesgo (Bad)", 0.89, temperature=0.2)
print(rapport_critique)